# CWRU Spectrogram Report Figure Generator

이 노트북은 CWRU `.mat` 진동 신호를 읽어서 **보고서/발표용 spectrogram figure**를 저장하는 파일이다.

- 축, 제목, colorbar 포함
- class별 대표 figure 저장
- 원본 파일별 figure 저장 가능
- 파일명 끝의 `_1`, `_2`는 **부하 조건(load condition)** 으로만 사용
- label은 `_1`, `_2`를 제거한 이름으로 사용

In [9]:
# 필요한 라이브러리
# 처음 실행하는 환경에서 에러가 나면 아래 주석을 해제해서 설치하면 된다.
# !pip install numpy scipy pandas matplotlib pillow tqdm

from pathlib import Path
import re
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.io import loadmat
from scipy import signal
from tqdm.auto import tqdm

## 1. 경로와 기본 설정

In [10]:
# =========================
# 사용자가 수정할 설정
# =========================

# .mat 파일들이 들어있는 폴더
MAT_DIR = Path("./mat_files")

# 결과 저장 폴더
OUTPUT_DIR = Path("./cwru_report_figures")

# 사용할 센서 채널
# DE: Drive End, FE: Fan End
CHANNEL = "DE"

# 샘플링 주파수
# 네 파일명이 48K 계열이면 48000으로 바꾸면 된다.
FS = 12000

# 보고서용 spectrogram에 사용할 segment 길이
SEGMENT_LENGTH = 4096

# STFT 설정
NPERSEG = 256
NOVERLAP = 128

# 각 파일에서 figure로 사용할 segment 시작 위치
# "middle"이면 신호 중앙부를 사용한다.
SEGMENT_POSITION = "middle"  # "start", "middle", "end"

# 저장 옵션
SAVE_EACH_FILE_FIGURE = True        # 파일별 figure 저장
SAVE_ONE_PER_LABEL_FIGURE = True    # label별 대표 figure 저장

# 컬러맵
CMAP = "viridis"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 2. 파일명에서 label과 load condition 추출

In [11]:
def label_from_filename(path: Path) -> str:
    # 마지막 _1, _2 같은 부하 번호를 제거해서 label을 만든다.
    # Fault_BALL_007_1.mat -> Fault_BALL_007
    # Normal_2.mat -> Normal
    return re.sub(r"_\d+$", "", path.stem)


def load_from_filename(path: Path) -> str:
    # 파일명 끝의 _1, _2를 부하 조건으로 추출한다.
    m = re.search(r"_(\d+)$", path.stem)
    return m.group(1) if m else "unknown"


def fault_type_from_label(label: str) -> str:
    # Fault_BALL_007 -> BALL
    # Fault_IR_014 -> IR
    # Normal -> Normal
    if label == "Normal":
        return "Normal"
    parts = label.split("_")
    return parts[1] if len(parts) >= 3 else "unknown"


def fault_size_from_label(label: str) -> str:
    # Fault_BALL_007 -> 007
    # Fault_IR_014 -> 014
    # Normal -> 000
    if label == "Normal":
        return "000"
    parts = label.split("_")
    return parts[2] if len(parts) >= 3 else "unknown"

## 3. `.mat` 파일에서 진동 신호 찾기

In [12]:
def find_signal_key(mat_dict: dict, channel: str = "DE") -> str:
    # CWRU .mat 내부에서 진동 신호 key를 자동으로 찾는다.
    # 우선 *_DE_time 또는 *_FE_time을 찾고,
    # 없으면 *_time 중 가장 긴 1D 신호를 사용한다.
    channel = channel.upper()
    candidates = []

    for k, v in mat_dict.items():
        if k.startswith("__"):
            continue
        if f"_{channel}_time" in k:
            arr = np.asarray(v).squeeze()
            if arr.ndim == 1 and arr.size > 100:
                candidates.append((k, arr.size))

    if not candidates:
        for k, v in mat_dict.items():
            if k.startswith("__"):
                continue
            if "_time" in k:
                arr = np.asarray(v).squeeze()
                if arr.ndim == 1 and arr.size > 100:
                    candidates.append((k, arr.size))

    if not candidates:
        raise ValueError("유효한 vibration time signal을 찾지 못했다.")

    candidates.sort(key=lambda x: x[1], reverse=True)
    return candidates[0][0]


def load_vibration_signal(mat_path: Path, channel: str = "DE"):
    mat = loadmat(mat_path)
    signal_key = find_signal_key(mat, channel)
    x = np.asarray(mat[signal_key]).squeeze().astype(np.float64)

    # 평균 제거 + 표준편차 정규화
    x = x - np.mean(x)
    std = np.std(x)
    if std > 1e-12:
        x = x / std

    return x, signal_key

## 4. 보고서용 spectrogram figure 생성 함수

In [13]:
def choose_segment(x: np.ndarray, segment_length: int, position: str = "middle"):
    if len(x) < segment_length:
        raise ValueError(f"신호 길이 {len(x)}가 segment_length {segment_length}보다 짧다.")

    if position == "start":
        start = 0
    elif position == "end":
        start = len(x) - segment_length
    elif position == "middle":
        start = (len(x) - segment_length) // 2
    else:
        raise ValueError("position은 'start', 'middle', 'end' 중 하나여야 한다.")

    seg = x[start:start + segment_length]
    return start, seg


def compute_spectrogram_db(seg, fs, nperseg, noverlap):
    f, t, S = signal.spectrogram(
        seg,
        fs=fs,
        window="hann",
        nperseg=nperseg,
        noverlap=noverlap,
        detrend=False,
        scaling="density",
        mode="magnitude",
    )
    S_db = 20 * np.log10(S + 1e-12)
    return f, t, S_db


def save_report_spectrogram(f, t, S_db, save_path: Path, title: str, cmap: str = "viridis"):
    fig, ax = plt.subplots(figsize=(7.2, 4.5), dpi=220)

    mesh = ax.pcolormesh(t, f, S_db, shading="gouraud", cmap=cmap)
    ax.set_title(title)
    ax.set_xlabel("Time within segment [s]")
    ax.set_ylabel("Frequency [Hz]")

    cbar = fig.colorbar(mesh, ax=ax)
    cbar.set_label("Magnitude [dB]")

    fig.tight_layout()
    save_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(save_path, bbox_inches="tight")
    plt.close(fig)

## 5. 전체 파일 처리

In [14]:
mat_files = sorted(MAT_DIR.glob("*.mat"))

if len(mat_files) == 0:
    raise FileNotFoundError(f"{MAT_DIR} 폴더에서 .mat 파일을 찾지 못했다.")

print(f"찾은 .mat 파일 수: {len(mat_files)}")
for p in mat_files:
    print(" -", p.name)

찾은 .mat 파일 수: 20
 - Fault_BALL_007_1.mat
 - Fault_BALL_007_2.mat
 - Fault_BALL_014_1.mat
 - Fault_BALL_014_2.mat
 - Fault_BALL_021_1.mat
 - Fault_BALL_021_2.mat
 - Fault_IR_007_1.mat
 - Fault_IR_007_2.mat
 - Fault_IR_014_1.mat
 - Fault_IR_014_2.mat
 - Fault_IR_021_1.mat
 - Fault_IR_021_2.mat
 - Fault_OR_007_1.mat
 - Fault_OR_007_2.mat
 - Fault_OR_014_1.mat
 - Fault_OR_014_2.mat
 - Fault_OR_021_1.mat
 - Fault_OR_021_2.mat
 - Normal_1.mat
 - Normal_2.mat


In [15]:
metadata = []
saved_label = set()

for mat_path in tqdm(mat_files):
    label = label_from_filename(mat_path)
    load_condition = load_from_filename(mat_path)
    fault_type = fault_type_from_label(label)
    fault_size = fault_size_from_label(label)

    x, signal_key = load_vibration_signal(mat_path, CHANNEL)
    start_sample, seg = choose_segment(x, SEGMENT_LENGTH, SEGMENT_POSITION)
    f, t, S_db = compute_spectrogram_db(seg, FS, NPERSEG, NOVERLAP)

    title = (
        f"{label} | load={load_condition} | channel={CHANNEL} | "
        f"source={mat_path.name}"
    )

    if SAVE_EACH_FILE_FIGURE:
        save_path = OUTPUT_DIR / "by_file" / label / f"{mat_path.stem}_spectrogram.png"
        save_report_spectrogram(f, t, S_db, save_path, title, CMAP)

    if SAVE_ONE_PER_LABEL_FIGURE and label not in saved_label:
        save_path = OUTPUT_DIR / "representative_by_label" / f"representative_{label}.png"
        save_report_spectrogram(f, t, S_db, save_path, title, CMAP)
        saved_label.add(label)

    metadata.append({
        "source_file": mat_path.name,
        "label": label,
        "load_condition": load_condition,
        "fault_type": fault_type,
        "fault_size": fault_size,
        "signal_key": signal_key,
        "channel": CHANNEL,
        "fs": FS,
        "start_sample": start_sample,
        "segment_length": SEGMENT_LENGTH,
        "nperseg": NPERSEG,
        "noverlap": NOVERLAP,
    })

metadata_df = pd.DataFrame(metadata)
metadata_df.to_csv(OUTPUT_DIR / "report_figure_metadata.csv", index=False, encoding="utf-8-sig")

metadata_df

  0%|          | 0/20 [00:00<?, ?it/s]

,source_file,label,load_condition,fault_type,fault_size,signal_key,channel,fs,start_sample,segment_length,nperseg,noverlap
0,Fault_BALL_007_1.mat,Fault_BALL_007,1,BALL,007,X123_DE_time,DE,12000,241644,4096,256,128
1,Fault_BALL_007_2.mat,Fault_BALL_007,2,BALL,007,X124_DE_time,DE,12000,241354,4096,256,128
2,Fault_BALL_014_1.mat,Fault_BALL_014,1,BALL,014,X190_DE_time,DE,12000,241064,4096,256,128
3,Fault_BALL_014_2.mat,Fault_BALL_014,2,BALL,014,X191_DE_time,DE,12000,241644,4096,256,128
4,Fault_BALL_021_1.mat,Fault_BALL_021,1,BALL,021,X227_DE_time,DE,12000,241354,4096,256,128
5,Fault_BALL_021_2.mat,Fault_BALL_021,2,BALL,021,X228_DE_time,DE,12000,241644,4096,256,128
6,Fault_IR_007_1.mat,Fault_IR_007,1,IR,007,X110_DE_time,DE,12000,241064,4096,256,128
7,Fault_IR_007_2.mat,Fault_IR_007,2,IR,007,X111_DE_time,DE,12000,240773,4096,256,128
8,Fault_IR_014_1.mat,Fault_IR_014,1,IR,014,X217_DE_time,DE,12000,242514,4096,256,128
9,Fault_IR_014_2.mat,Fault_IR_014,2,IR,014,X176_DE_time,DE,12000,241934,4096,256,128


## 6. 보고서에 쓸 수 있는 설명 문장 예시

> 본 연구에서는 CWRU 베어링 진동 신호를 시간-주파수 영역에서 분석하기 위해 STFT 기반 spectrogram으로 변환하였다. 각 `.mat` 파일의 진동 신호는 평균 제거 및 표준편차 정규화를 거친 뒤, 고정 길이 segment로 분할하였다. 보고서용 figure는 각 클래스의 대표 segment에 대해 시간축, 주파수축, dB 단위 magnitude colorbar를 포함하도록 저장하였다. 파일명 끝의 `_1`, `_2`는 부하 조건을 의미하므로, 분류 레이블에서는 제외하고 별도의 load condition metadata로 관리하였다.